## Файл для преобразования скачанного датасета в формат YOLO


In [1]:
import xml.etree.ElementTree as ET
from pathlib import Path

In [2]:
XML_DIR  = Path("DoReMi_v1/Parsed_by_page_omr_xml")       
IMG_DIR  = Path("DoReMi_v1/Images")      
OUT_DIR  = Path("dataset2")      
# STAFF_DIR = Path("dataset/staff_lines/all")

IMG_W, IMG_H = 2475, 3504           

SCALE        = 0.5    
TILE_SIZE    = 640   
TILE_STRIDE  = 400  

SCALED_W = int(IMG_W * SCALE)   
SCALED_H = int(IMG_H * SCALE) 

# Минимальная доля площади объекта, которая должна попасть в тайл,
# чтобы он был включён в разметку. 0.3 = 30%.
MIN_VISIBILITY = 0.3

MIN_BLACK_PIXELS = 500   # минимум чёрных пикселей в тайле


In [3]:
# Ключ = ClassName в XML, значение = индекс класса YOLO
KEEP_CLASSES: dict[str, int] = {
    # Головки нот
    "noteheadBlack":        0,
    "noteheadHalf":         1,
    "noteheadWhole":        1,
    # Знаки альтерации
    "accidentalSharp":      2,
    "accidentalFlat":       3,
    "accidentalNatural":    4,
    "accidentalDoubleSharp":        5,
    "accidentalDoubleFlat":         6,
    # Скрипичный ключ
    "gClef":                7,
}

In [ ]:
OTHER_CLASSES: dict[str, int] = {
    # class_name → class_id

    # [8] dot — маленькие заполненные точки ≈ notehead_filled
    "augmentationDot":          8,
    "articStaccatoAbove":       8,
    "articStaccatoBelow":       8,
    "articStaccatissimoAbove":  8,
    "articStaccatissimoBelow":  8,

    # [9] stem — вертикальные штили
    "stem":                     9,
    "systemicBarline":          9,
    "barline":                  9,

    # [10] beam — горизонтальные балки ≈ ledger_line
    "beam":                     10,

    # [11] rest1 — прямоугольные паузы ≈ sharp/natural по форме
    "restWhole":                11,
    "restHalf":                 11,

    # [12] rest2 - 
    "restQuarter":              12,
    "rest8th":                  12,
    "rest15th":                 12,
    "rest32nd":                 12,

    # [13] other_clef — другие ключи ≈ gClef
    "fClef":                    13,
    "cClef":                    13,
    "unpitchedPercussionClef1": 13,

    # [14] curve — лиги, тирэ, крещендо (изогнутые линии)
    "tie":                      14,
    "slur":                     14,
    "gradualDynamic":           14,
    "tupletBracket":            14,

    # [15] other — всё остальное с низким риском (маленькой частотой)
    "flag8thUp":                15,
    "flag8thDown":              15,
    "flag15thUp":               15,
    "flag15thDown":             15,
    "flag32ndUp":               15,
    "flag32ndDown":             15,
    "articAccentAbove":         15,
    "articAccentBelow":         15,
    "articTenutoAbove":         15,
    "articTenutoBelow":         15,
    "articMarcatoAbove":        15,
    "articMarcatoBelow":        15,
    "dynamicPiano":             15,
    "dynamicForte":             15,
    "dynamicFF":                15,
    "dynamicFFF":               15,
    "dynamicPP":                15,
    "dynamicPPP":               15,
    "dynamicMF":                15,
    "dynamicMP":                15,
    "dynamicFortePiano":        15,
    "dynamicForzando":          15,
    "dynamicSforzato":          15,
    "dynamicText":              15,
    "timeSig2":                 15,
    "timeSig3":                 15,
    "timeSig4":                 15,
    "timeSig6":                 15,
    "timeSig8":                 15,
    "timeSigCommon":            15,
    "timeSigCutCommon":         15,
    "timeSignatureComponent":   15,
    "ornamentTrill":            15,
    "ornamentMordent":          15,
    "ornamentTurn":             15,
    "wiggleTrill":              15,
    "tupletText":               15,
}

In [5]:
CLASS_NAMES: dict[int, str] = {
    # Целевые
    0: "notehead_filled",
    1: "notehead_open",
    2: "sharp",
    3: "flat",
    4: "natural",
    5: "double_sharp",
    6: "double_flat",
    7: "clef",
    # Вспомогательные
    8:  "dot",
    9: "stem",
    10: "beam",
    11: "rest1",
    12: "rest2",
    13: "other_clef",
    14: "curve",
    15: "other",
}

In [6]:
ALIASES: dict[str, str] = {
    "notehead-full":        "noteheadBlack",
    "notehead-empty":       "noteheadHalf",
    "sharp":                "accidentalSharp",
    "flat":                 "accidentalFlat",
    "natural":              "accidentalNatural",
    "double-sharp":         "accidentalDoubleSharp",
    "double-flat":          "accidentalDoubleFlat",
    "leger-line":           "legerLine",
    "g-clef":               "gClef",
    # DeepScores
    "noteheadBlackOnLine":  "noteheadBlack",
    "noteheadBlackInSpace": "noteheadBlack",
    "noteheadHalfOnLine":   "noteheadHalf",
    "noteheadHalfInSpace":  "noteheadHalf",
    "noteheadWholeOnLine":  "noteheadWhole",
    "noteheadWholeInSpace": "noteheadWhole",
}

In [7]:

def parse_xml(xml_path: Path) -> list[dict]:
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError as e:
        print(f"  [ERROR] {xml_path.stem}: {e}")
        return []

    objects: list[dict] = []
    staff_ys: list[int] = []

    for node in root.iter("Node"):
        raw = node.findtext("ClassName", "").strip()
        class_name = ALIASES.get(raw, raw)

        if class_name == "kStaffLine":
            try:
                staff_ys.append(int(node.findtext("Top")))
            except (TypeError, ValueError):
                pass
            continue
        if class_name in KEEP_CLASSES:
            class_id = KEEP_CLASSES[class_name]
        elif class_name in OTHER_CLASSES:
            class_id = OTHER_CLASSES[class_name]
        else:
            continue
        try:
            top, left = int(node.findtext("Top")), int(node.findtext("Left"))
            w, h      = int(node.findtext("Width")), int(node.findtext("Height"))
        except (TypeError, ValueError):
            print(f"  [WARN] {xml_path.stem}: неверные координаты, узел пропущен")
            continue
        if w > 0 and h > 0:
            objects.append({"class_id": class_id,
                            "top": top, "left": left, "w": w, "h": h})
    return objects, sorted(staff_ys)


In [8]:
def scale_objects(objects: list[dict], scale: float = SCALE) -> list[dict]:
    """
    Умножает пиксельные координаты на scale.
    w/h минимум 1 пиксель после масштабирования.
    """
    out = []
    for o in objects:
        out.append({
            "class_id": o["class_id"],
            "top":  int(o["top"]  * scale),
            "left": int(o["left"] * scale),
            "w":    max(1, int(o["w"] * scale)),
            "h":    max(1, int(o["h"] * scale)),
        })
    return out

Возвращает список начальных координат тайлов по одной оси.
Последний тайл прижимается к правому/нижнему краю изображения,
чтобы гарантировать полное покрытие без выхода за границу.

In [9]:
def tile_offsets(img_size: int, tile: int = TILE_SIZE, stride: int = TILE_STRIDE) -> list[int]:
    
    offsets = list(range(0, img_size - tile + 1, stride))
    # Добавляем крайний тайл только если он не совпадает с последним
    last = img_size - tile
    if last > 0 and (not offsets or offsets[-1] < last):
        offsets.append(last)
    return offsets


def objects_for_tile(
    objects: list[dict],
    tx: int,
    ty: int,
    tile: int  = TILE_SIZE,
    min_vis: float = MIN_VISIBILITY,
) -> list[str]:
    lines: list[str] = []
    tx2, ty2 = tx + tile, ty + tile

    for o in objects:
        ox1, oy1 = o["left"], o["top"]
        ox2, oy2 = ox1 + o["w"], oy1 + o["h"]

        ix1 = max(ox1, tx);  ix2 = min(ox2, tx2)
        iy1 = max(oy1, ty);  iy2 = min(oy2, ty2)

        if ix2 <= ix1 or iy2 <= iy1:
            continue  

        inter_area = (ix2 - ix1) * (iy2 - iy1)
        obj_area   = o["w"] * o["h"]
        if inter_area / obj_area < min_vis:
            continue 

        cx = ((ix1 + ix2) / 2 - tx) / tile
        cy = ((iy1 + iy2) / 2 - ty) / tile
        bw = (ix2 - ix1) / tile
        bh = (iy2 - iy1) / tile

        lines.append(f"{o['class_id']} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

    return lines

In [10]:
import re

def xml_stem_to_img_stem(xml_stem: str) -> str | None:
    m = re.fullmatch(
        r"Parsed_(.+)-layout-\d+-muscima_Page_(\d+)",
        xml_stem
    )
    if not m:
        print(f"  [WARN] Не удалось распарсить имя файла: {xml_stem!r}")
        return None

    piece_name = m.group(1)          
    page_num   = int(m.group(2))   
    return f"{piece_name}-{page_num:03d}"

def group_staves(staff_ys: list[int], gap: int = 30) -> list[list[int]]:
    if not staff_ys:
        return []
    staves, current = [], [staff_ys[0]]
    for y in staff_ys[1:]:
        if y - current[-1] > gap:
            staves.append(current)
            current = []
        current.append(y)
    staves.append(current)
    return staves

In [11]:
import json
import cv2
import numpy as np

def convert_one(xml_path: Path, split:     str, out_dir: Path) -> dict[int, int]:
    
    new_stem = xml_stem_to_img_stem(xml_path.stem)
    if new_stem is None:
        return {}
    
    img_out_dir   = out_dir / "images"      / split
    lbl_out_dir   = out_dir / "labels"      / split
    staff_out_dir = out_dir / "staff_lines" / split
    for d in (img_out_dir, lbl_out_dir, staff_out_dir):
        d.mkdir(parents=True, exist_ok=True)

    objects, staff_ys = parse_xml(xml_path)
    objects = scale_objects(objects, SCALE)
    img_path = IMG_DIR / (new_stem + '.png')

    if not img_path.exists():
        print(f"  [WARN] Изображение не найдено: {img_path}")
        img = None
    else:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"  [WARN] Не удалось загрузить: {img_path}")
        else:
            img = cv2.resize(
                img,
                (SCALED_W, SCALED_H),
                interpolation=cv2.INTER_AREA,  
            )

    xs = tile_offsets(SCALED_W, TILE_SIZE, TILE_STRIDE)
    ys = tile_offsets(SCALED_H, TILE_SIZE, TILE_STRIDE)

    tile_count = 0
    for ty in ys:
        for tx in xs:
            tile_name = f"{new_stem}_x{tx:04d}_y{ty:04d}"
            if img is not None:
                tile_img = img[ty:ty + TILE_SIZE, tx:tx + TILE_SIZE]
                gray = cv2.cvtColor(tile_img, cv2.COLOR_BGR2GRAY)
                black_pixels = np.sum(gray < 128)
                if black_pixels < MIN_BLACK_PIXELS:
                    continue
                cv2.imwrite(str(img_out_dir / (tile_name + ".jpg")), tile_img,
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
                
            yolo_lines = objects_for_tile(objects, tx, ty)
            lbl_file = lbl_out_dir / (tile_name + ".txt")
            with open(lbl_file, "w", encoding="utf-8") as f:
                if yolo_lines:
                    f.write("\n".join(yolo_lines) + "\n")

            

            tile_count += 1
    staves = group_staves([int(y * SCALE) for y in staff_ys])
    staff_data = {
        "image":        new_stem,
        "scaled_w":     SCALED_W,
        "scaled_h":     SCALED_H,
        "stave_count":  len(staves),
        "staves": [
            {
                "lines_px":   s,
                "lines_norm": [round(y / SCALED_H, 6) for y in s]
            }
            for s in staves
        ]
    }
    with open(staff_out_dir / (new_stem + ".json"), "w", encoding="utf-8") as f:
        json.dump(staff_data, f, ensure_ascii=False, indent=2)
    counts: dict[int, int] = {}
    for o in objects:
        counts[o["class_id"]] = counts.get(o["class_id"], 0) + 1

    print(f"  {new_stem}: {len(objects)} объектов → {tile_count} тайлов")
    return counts

In [12]:
def assign_split(stem: str) -> str:
    TRAIN_RATIO = 0.8
    VAL_RATIO   = 0.1
    h = int(abs(hash(stem))) % 100
    if h < int(TRAIN_RATIO * 100):
        return "train"
    elif h < int((TRAIN_RATIO + VAL_RATIO) * 100):
        return "val"
    return "test"

Для одного изображения:


In [13]:

SINGLE_XML = XML_DIR / "Parsed_accidental tucking-layout-0-muscima_Page_1.xml"  

stem   = xml_stem_to_img_stem(SINGLE_XML.stem)
split  = assign_split(stem)
counts = convert_one(SINGLE_XML, split, OUT_DIR)

print("\nОбъектов по классам:")
for cid, name in CLASS_NAMES.items():
    print(f"  [{cid}] {name:<16} {counts.get(cid, 0)}")

  accidental tucking-001: 94 объектов → 5 тайлов

Объектов по классам:
  [0] notehead_filled  30
  [1] notehead_open    0
  [2] sharp            9
  [3] flat             0
  [4] natural          2
  [5] double_sharp     0
  [6] double_flat      0
  [7] clef             1
  [8] dot              0
  [9] stem             21
  [10] beam             0
  [11] rest1            0
  [12] rest2            9
  [13] other_clef       0
  [14] curve            0
  [15] other            22


Для всех:


In [14]:

xml_files = sorted(XML_DIR.glob("*.xml"))
print(f"Найдено файлов: {len(xml_files)}\n")

total_counts: dict[int, int] = {}
split_counts = {"train": 0, "val": 0, "test": 0}

for xml_file in xml_files:
    stem  = xml_stem_to_img_stem(xml_file.stem)
    if stem is None:
        continue
    split = assign_split(stem)
    split_counts[split] += 1

    counts = convert_one(xml_file,split, OUT_DIR)
    for cid, n in counts.items():
        total_counts[cid] = total_counts.get(cid, 0) + n
    print()

total = sum(total_counts.values())
scale = max(1, total // 400)
print("─" * 50)
print(f"Итого объектов: {total}")
print(f"Разбивка страниц: train={split_counts['train']}, "
      f"val={split_counts['val']}, test={split_counts['test']}")
print("Распределение:")
for cid, name in CLASS_NAMES.items():
    count = total_counts.get(cid, 0)
    bar   = "█" * min(30, count // scale)
    print(f"  [{cid}] {name:<16} {count:>7}  {bar}")
print("─" * 50)

Найдено файлов: 5218

  Alkan - Posement-001: 209 объектов → 6 тайлов

  Alkan - Posement-010: 212 объектов → 3 тайлов

  Alkan - Posement-011: 230 объектов → 3 тайлов

  Alkan - Posement-012: 219 объектов → 4 тайлов

  Alkan - Posement-013: 216 объектов → 3 тайлов

  Alkan - Posement-014: 216 объектов → 6 тайлов

  Alkan - Posement-015: 176 объектов → 6 тайлов

  Alkan - Posement-016: 200 объектов → 3 тайлов

  Alkan - Posement-017: 205 объектов → 3 тайлов

  Alkan - Posement-018: 224 объектов → 3 тайлов

  Alkan - Posement-019: 222 объектов → 6 тайлов

  Alkan - Posement-002: 207 объектов → 3 тайлов

  Alkan - Posement-020: 233 объектов → 3 тайлов

  Alkan - Posement-021: 223 объектов → 3 тайлов

  Alkan - Posement-022: 230 объектов → 3 тайлов

  Alkan - Posement-023: 218 объектов → 3 тайлов

  Alkan - Posement-024: 249 объектов → 3 тайлов

  Alkan - Posement-025: 243 объектов → 3 тайлов

  Alkan - Posement-026: 234 объектов → 3 тайлов

  Alkan - Posement-027: 234 объектов → 6 тайлов

In [15]:
for split in ("train", "val", "test"):
    imgs = list((OUT_DIR / "images" / split).glob("*.jpg"))
    lbls = list((OUT_DIR / "labels" / split).glob("*.txt"))
    print(f"{split:5s}:  {len(imgs):5d} изображений   {len(lbls):5d} файлов разметки")

train:  16796 изображений   16796 файлов разметки
val  :   3288 изображений    3288 файлов разметки
test :   3325 изображений    3325 файлов разметки
